# SOXX 官方 NAV 与溢折价下载来源预检

一次固定官方下载；只检查原始工作簿结构、日期覆盖与时点资格，不读取未来收益。


In [1]:
import sys,json,hashlib,re,datetime,collections,math,tarfile,tempfile,subprocess,shlex
from pathlib import Path
from xml.etree import ElementTree as ET
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import _settings,nas_api_call
card=root/'research/stage1_soxx_official_premium_discount_source_20261023';snap=root/'data/snapshots/stage1_soxx_official_premium_discount_source_20261023';raw=snap/'fund_download.bin'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
plan=json.loads((card/'plan.json').read_text());t=json.loads((card/'transport_result.json').read_text());a=t['attempt']
assert plan['frozen'] and t['plan_sha256']==sha(card/'plan.json') and t['GETs_consumed']==1 and t['retries']==0 and a['url']==plan['exact_GET_url']
assert a['http_status']==200 and a['body_sha256']==sha(raw) and a['body_bytes']==raw.stat().st_size and a['content_type']=='application/vnd.ms-excel'
original=raw.read_text(encoding='utf-8');repaired,amp_count=re.subn(r'&(?!amp;|lt;|gt;|quot;|apos;|#[0-9]+;|#x[0-9A-Fa-f]+;)', '&amp;',original)
assert amp_count==6 and repaired!=original
xml=ET.fromstring(repaired);ns={'s':'urn:schemas-microsoft-com:office:spreadsheet'};key='{urn:schemas-microsoft-com:office:spreadsheet}Name'
sheets={w.attrib[key]:w.findall('.//s:Row',ns) for w in xml.findall('s:Worksheet',ns)}
assert list(sheets)==['Disclaimers','Holdings','Historical','Performance','Distributions']
rowvals=lambda row:[c.findtext('s:Data',default='',namespaces=ns) for c in row.findall('s:Cell',ns)]
header=rowvals(sheets['Historical'][0]);assert header==['As Of','NAV per Share','Ex-Dividends','Shares Outstanding']
assert max(len(rowvals(row)) for row in sheets['Historical'])==4
nav=[]
for row in sheets['Historical'][1:]:
 vals=rowvals(row);d=datetime.datetime.strptime(vals[0],'%b %d, %Y').date();v=float(vals[1]);assert math.isfinite(v) and v>0;nav.append((d,v))
assert len(nav)==8937 and len({d for d,v in nav})==len(nav)
assert nav[0][0]==datetime.date(2026,10,21) and nav[-1][0]==datetime.date(2001,7,10)
nav=sorted(nav);lookup=dict(nav);years=collections.Counter(d.year for d,v in nav)
weekend={str(y):sum(d.year==y and d.weekday()>=5 for d,v in nav) for y in (2011,2018,2023,2024,2025,2026)}
weekend_unchanged={str(y):sum(d.year==y and d.weekday()>=5 and lookup.get(d-datetime.timedelta(days=1))==v for d,v in nav) for y in (2011,2018,2023,2024,2025,2026)}
audit={'source_sha256':sha(raw),'http_status':200,'content_type':a['content_type'],'parser_repaired_unescaped_ampersands':amp_count,'raw_bytes_preserved':True,'sheet_rows':{k:len(v) for k,v in sheets.items()},'historical_header':header,'historical_daily_market_price_column':False,'historical_premium_discount_column':False,'historical_NAV_rows':len(nav),'first_date':nav[0][0].isoformat(),'last_date':nav[-1][0].isoformat(),'duplicate_dates':0,'nonpositive_NAV_rows':0,'calendar_rows_2011_2022':sum(years[y] for y in range(2011,2023)),'calendar_rows_2023_2026':sum(years[y] for y in range(2023,2027)),'year_rows':{str(y):years[y] for y in range(2011,2027)},'weekend_rows_selected_years':weekend,'weekend_equal_prior_day_selected_years':weekend_unchanged,'first_publication_time_qualified':False,'current_revision_only':True,'SOXX_future_outcomes_read':0,'model_fits':0,'native_accounts':0}
(card/'source_audit.json').write_text(json.dumps(audit,indent=2)+'\n')
report={'decision':'official_fund_download_has_historical_NAV_but_no_dated_market_price_or_premium_series','source_audit':'source_audit.json','plan_sha256':sha(card/'plan.json'),'source_sha256':sha(raw),'direct_premium_discount_source_qualified':False,'historical_NAV_source_observed':True,'historical_first_receipt_qualified':False,'Qlib_factor_map_allowed':False,'new_model_fits':0,'new_native_accounts':0}
(card/'result.json').write_text(json.dumps(report,indent=2)+'\n')
settings=_settings(root);members=[card/'plan.json',card/'transport_result.json',card/'source_audit.json',raw]
with tempfile.TemporaryDirectory(prefix='soxx-nav-source-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for p in members:out.add(p,arcname=str(p.relative_to(root)))
 digest=sha(archive);response=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research');receipt=response['artifact']
 assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size
 nas_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 restored=Path(tmp)/'restored.tar.gz'
 with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(nas_path)],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(restored)==digest
 with tarfile.open(restored,'r:gz') as z:
  names={m.name:m for m in z.getmembers() if m.isfile()};assert set(names)=={str(p.relative_to(root)) for p in members}
  for p in members:assert hashlib.sha256(z.extractfile(names[str(p.relative_to(root))]).read()).hexdigest()==sha(p)
source_archive={'status':'official_workbook_source_NAS_scoped_restore_verified','sha256':digest,'size_bytes':receipt['size_bytes'],'nas_path':nas_path,'members':{str(p.relative_to(root)):sha(p) for p in members},'verified':True}
(card/'nas_source_archive.json').write_text(json.dumps(source_archive,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':report['plan_sha256'],'result':'result.json','source_audit':'source_audit.json','source_archive':'nas_source_archive.json','model_fits':0,'native_accounts':0,'SOXX_future_outcome_reads':0,'Qlib_Recorder':None,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]},'limitations':['The current download has NAV but lacks the other component of premium/discount.','2011-2023 calendar weekends carry the prior NAV; 2024 mixes calendar and trading-day rows, requiring native exchange-calendar alignment if NAV is later used.','Current revised workbook does not prove first-publication time or historical vintage.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps({'sheets':audit['sheet_rows'],'NAV_rows':len(nav),'has_direct_premium':False,'nas_sha256':digest}))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{"sheets": {"Disclaimers": 14, "Holdings": 42, "Historical": 8938, "Performance": 308, "Distributions": 94}, "NAV_rows": 8937, "has_direct_premium": false, "nas_sha256": "a75d8e98023daa4e3fb5d78dd1f7a9c1e9c67194b5fb6215aa5be2bc14620a2c"}


# SOXX 官方下载：历史 NAV 与溢折价来源门

iShares 官方 SOXX 页面显示当日 NAV、收盘价和溢折价，并提供“Data Download”。按冻结计划只下载一次官方文件（HTTP 200，原字节 3,347,697）。文件实际是 SpreadsheetML XML，五张表中 `Historical` 有 8,937 条 2001-07-10 至 2026-10-21 的日期和正值 NAV；列仅为 `As Of`、`NAV per Share`、`Ex-Dividends`、`Shares Outstanding`，**没有历史市场价或溢折价列**。因此这份下载本身不构成预设特征的完整来源。

历史表的日期粒度也不统一：2011、2018、2023 年分别列出 365、365、365 个日历日，周末 NAV 全部沿用前日；2024 年 275 行且有 19 个周末，2025 年 250 行无周末。后续若使用 NAV，必须用原生交易日历去除重复日，并验证分拆调整、原始发布时刻和修订版本。当前下载不能证明这些日值在原 Stage1 的 09:47 决策前曾按此数值可得。

原件包含六个未转义的 `&`，其中两处在 URL、四处在文字中；只在临时解析副本中转义。原始字节、传输收据、结构审计和冻结计划均已按哈希归档 Yanxuan NAS，并完成限定恢复。0 未来 SOXX 收益读取、0 拟合、0 账户；未建 Qlib 因子或 Recorder。

**决定：关闭“该下载直接提供历史溢折价”的来源版本，不推广。** NAV 本身仍可作为后续独立来源支线，但须另找同口径历史市场价或发行人直接发布的溢折价及其首次可用时间，另行冻结拼接和信息检验。
